In [ ]:
# MOTOR-PK / COLAB SINGLE CELL / OFFLINE GPU PREFLIGHT
import contextlib, hashlib, json, os, platform, re, shutil, subprocess, sys, time, uuid
from datetime import datetime, timezone

CONFIG = json.loads('{"allowed_gpu_pattern": "(?i)\\\\b(?:nvidia\\\\s+)?(?:tesla\\\\s+)?(?:t4|l4|a100)\\\\b", "artifact_file_name": "motor_pk_gpu_preflight-motor-pk-colab-20260821T230715Z-0fa5b8ae.ipynb", "artifact_path": "notebooks/motor_pk_gpu_preflight-motor-pk-colab-20260821T230715Z-0fa5b8ae.ipynb", "artifact_repository": "felipesp1983/motor-pk-colab-artifacts", "data_policy": "synthetic-only-no-poker-data", "direct_colab_url": "https://colab.research.google.com/github/FELIPEACASTRO/mr_poker/blob/codex/colab-gpu-preflight-2d0a4c68/infra/notebooks/motor_pk_gpu_preflight.ipynb", "drive_enabled": false, "github_mirror_url": "https://github.com/FELIPEACASTRO/mr_poker/blob/codex/colab-gpu-preflight-2d0a4c68/infra/notebooks/motor_pk_gpu_preflight.ipynb", "gpu_preference_order": ["A100", "L4", "T4"], "gpu_roles": {"A100": "high_capacity_training_after_data_and_budget_gate", "L4": "recommended_default_for_preflight_and_bounded_experiments", "T4": "fallback_for_preflight_and_small_experiments_only"}, "gpu_selection_mode": "L4_metadata_preference_with_explicit_A100_or_T4_runtime_fallback", "matrix_iterations": 3, "matrix_size": 1024, "minimum_free_disk_gib": 20, "minimum_system_ram_gib": 8, "minimum_vram_gib": 14, "notebook_id": "motor-pk-colab-20260821T230715Z-0fa5b8ae", "officially_tested_gpu_models": ["A100", "L4", "T4"], "project": "MOTOR-PK", "publication_status": "github_mirror_verified", "purpose": "offline-gpu-preflight", "recommended_gpu": "L4", "requested_accelerator": "GPU", "requested_gpu_type": "L4", "requested_tpu_type": null, "run_id": "motor-pk-colab-20260821T230715Z-0fa5b8ae", "telemetry_flush_interval_seconds": 15, "telemetry_flush_max_events": 25, "telemetry_repository": "felipesp1983/motor-pk-colab-telemetry", "telemetry_retry_attempts": 3, "telemetry_schema_version": 2, "telemetry_secret_name": "HF_TELEMETRY_WRITER_TOKEN", "version": "0.4.1"}')
STARTED = time.monotonic()
TRACES = []
TELEMETRY = None
SECRET_FIELD = re.compile(r"(secret|token|key|password|authorization|cookie)", re.I)
SECRET_VALUE = re.compile(r"(?i)(?:\b(?:hf|ghp|github_pat|sk)_[A-Za-z0-9_-]{8,}\b|\bBearer\s+[A-Za-z0-9._-]{8,}\b)")
MAX_TRACE_DEPTH = 4

def _safe(value, depth=0):
    if depth > MAX_TRACE_DEPTH:
        return "<depth-limit>"
    if isinstance(value, dict):
        return {str(key): "<redacted>" if SECRET_FIELD.search(str(key)) else _safe(item, depth + 1) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_safe(item, depth + 1) for item in value[:20]]
    if isinstance(value, str):
        if SECRET_VALUE.search(value):
            return "<redacted>"
        return value[:300]
    if isinstance(value, (int, float, bool, type(None))):
        return value
    return type(value).__name__

def trace(stage, event, level="INFO", **details):
    record = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "notebook_id": CONFIG["notebook_id"],
        "run_id": CONFIG["run_id"],
        "version": CONFIG["version"],
        "project": CONFIG["project"],
        "stage": stage,
        "event": event,
        "level": level,
        "elapsed_s": round(time.monotonic() - STARTED, 3),
        "sequence": len(TRACES) + 1,
        "details": _safe(details),
    }
    TRACES.append(record)
    print("COLAB_TRACE " + json.dumps(record, ensure_ascii=False, sort_keys=True))
    if TELEMETRY is not None:
        TELEMETRY.emit(record, force=event in {"completed", "failed"})
    return record

@contextlib.contextmanager
def traced(stage):
    trace(stage, "started")
    try:
        yield
    except Exception as exc:
        trace(stage, "failed", "ERROR", error_type=type(exc).__name__)
        raise
    else:
        trace(stage, "completed")

class MetricPredictor:
    """Detector de tendência; não constitui prova causal de desempenho."""
    def __init__(self, mode="min", window=8, epsilon=1e-9):
        self.mode, self.window, self.epsilon, self.points = mode, window, epsilon, []
    def observe(self, step, value):
        self.points.append((float(step), float(value)))
        points = self.points[-self.window:]
        if len(points) < 3:
            return {"state": "insufficient_data", "n": len(points), "reason": "need_at_least_3_observations"}
        xs, ys = zip(*points)
        xbar, ybar = sum(xs) / len(xs), sum(ys) / len(ys)
        denominator = sum((x - xbar) ** 2 for x in xs)
        slope = 0.0 if denominator == 0 else sum((x - xbar) * (y - ybar) for x, y in points) / denominator
        signed_slope = slope if self.mode == "min" else -slope
        state = "converging" if signed_slope < -self.epsilon else "diverging" if signed_slope > self.epsilon else "plateau"
        return {"state": state, "n": len(points), "slope_per_step": slope, "latest": ys[-1], "predicted_next": ys[-1] + slope, "reason": "linear_recent_trend_not_causal_proof"}

def _memory_gib():
    return round((os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")) / (1024 ** 3), 2)

def _gpu_snapshot():
    query = ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"]
    completed = subprocess.run(query, capture_output=True, check=False, text=True, timeout=20)
    lines = [line.strip() for line in completed.stdout.splitlines() if line.strip()]
    if completed.returncode != 0 or not lines:
        raise RuntimeError("gpu_runtime_not_allocated")
    gpus = []
    for line in lines:
        fields = [field.strip() for field in line.split(",")]
        if len(fields) != 3:
            raise RuntimeError("unexpected_nvidia_smi_output")
        gpus.append({"name": fields[0], "memory_total_mib": int(fields[1]), "driver_version": fields[2]})
    return gpus

def _gpu_label(name):
    normalized = name.upper()
    for label in CONFIG["officially_tested_gpu_models"]:
        if label in normalized:
            return label
    raise RuntimeError("allocated_gpu_is_outside_documented_model_set")

def verify_requested_runtime():
    gpus = _gpu_snapshot()
    allowed = re.compile(CONFIG["allowed_gpu_pattern"])
    if not all(allowed.search(gpu["name"]) for gpu in gpus):
        raise RuntimeError("allocated_gpu_is_outside_allowlist")
    primary = gpus[0]
    primary_label = _gpu_label(primary["name"])
    minimum_vram_mib = int(CONFIG["minimum_vram_gib"] * 1024)
    if primary["memory_total_mib"] < minimum_vram_mib:
        raise RuntimeError("allocated_gpu_has_insufficient_vram")
    system_ram_gib = _memory_gib()
    if system_ram_gib < CONFIG["minimum_system_ram_gib"]:
        raise RuntimeError("allocated_runtime_has_insufficient_system_ram")
    free_disk_gib = round(shutil.disk_usage("/content").free / (1024 ** 3), 2)
    if free_disk_gib < CONFIG["minimum_free_disk_gib"]:
        raise RuntimeError("allocated_runtime_has_insufficient_free_disk")
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("pytorch_cuda_not_available")
    device = torch.device("cuda:0")
    properties = torch.cuda.get_device_properties(device)
    snapshot = {
        "gpus": gpus,
        "system_ram_gib": system_ram_gib,
        "free_disk_gib": free_disk_gib,
        "torch_version": torch.__version__,
        "cuda_runtime": torch.version.cuda,
        "torch_device_name": properties.name,
        "torch_device_memory_mib": round(properties.total_memory / (1024 ** 2), 2),
        "python": platform.python_version(),
        "allocated_gpu_model": primary_label,
        "allocated_gpu_role": CONFIG["gpu_roles"][primary_label],
        "recommended_gpu": CONFIG["recommended_gpu"],
        "preference_order": CONFIG["gpu_preference_order"],
        "selection_mode": CONFIG["gpu_selection_mode"],
    }
    trace("preflight", "accelerator_verified", requested="GPU", hardware=snapshot)
    return snapshot

def load_secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception as exc:
        raise RuntimeError("required_colab_secret_unavailable:" + type(exc).__name__) from None
    if not value:
        raise RuntimeError("required_colab_secret_unavailable")
    return value

def ensure_huggingface_hub():
    try:
        from huggingface_hub import CommitOperationAdd, HfApi
    except ImportError:
        trace("telemetry", "dependency_install_started", package="huggingface_hub==0.34.0")
        command = [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "huggingface_hub==0.34.0"]
        completed = subprocess.run(command, capture_output=True, check=False, text=True, timeout=180)
        if completed.returncode != 0:
            raise RuntimeError("huggingface_hub_install_failed")
        from huggingface_hub import CommitOperationAdd, HfApi
    return CommitOperationAdd, HfApi

class HFTelemetry:
    """Entrega confirmada de telemetria sanitizada; token nunca é serializado."""
    EVENT_FIELDS = ("timestamp_utc", "notebook_id", "run_id", "version", "project", "stage", "event", "level", "elapsed_s", "sequence")
    def __init__(self, repo_id, token, commit_operation_add, hf_api, spool_root=None):
        self.repo_id = repo_id
        self.token = token
        self.commit_operation_add = commit_operation_add
        self.api = hf_api(token=token)
        self.buffer, self.chunks, self.pending = [], [], None
        self.failed_attempts, self.finalized = 0, False
        self.last_successful_flush_monotonic = time.monotonic()
        self.spool_root = os.path.abspath(spool_root or os.path.join("/content", ".motor-pk-telemetry-spool", CONFIG["run_id"]))
        if spool_root is None and not self.spool_root.startswith(os.path.abspath("/content") + os.sep):
            raise RuntimeError("telemetry_spool_outside_content")
        os.makedirs(self.spool_root, exist_ok=True)
    def _event(self, record):
        event = {field: record.get(field) for field in self.EVENT_FIELDS}
        event["details"] = _safe(record.get("details", {}))
        event.update({"schema_version": CONFIG["telemetry_schema_version"], "event_id": uuid.uuid4().hex, "trace_id": CONFIG["run_id"]})
        return event
    def emit(self, record, force=False):
        self.buffer.append(self._event(record))
        due_to_time = time.monotonic() - self.last_successful_flush_monotonic >= CONFIG["telemetry_flush_interval_seconds"]
        if force or due_to_time or len(self.buffer) >= CONFIG["telemetry_flush_max_events"]:
            return self.flush()
        return True
    def _write_spool(self, pending):
        path = os.path.join(self.spool_root, pending["chunk"]["sha256"] + ".jsonl")
        temporary = path + ".partial"
        try:
            with open(temporary, "wb") as handle:
                handle.write(pending["data"])
            os.replace(temporary, path)
        except Exception as exc:
            print("COLAB_TELEMETRY " + json.dumps({"event": "spool_write_failed", "error_type": type(exc).__name__}, sort_keys=True))
        else:
            pending["spool_path"] = path
    def _cleanup_spool(self, pending):
        path = pending.get("spool_path")
        try:
            if path and os.path.isfile(path):
                os.remove(path)
            if os.path.isdir(self.spool_root) and not os.listdir(self.spool_root):
                os.rmdir(self.spool_root)
        except OSError:
            pass
    def _prepare_pending(self):
        if self.pending is not None:
            return self.pending
        if not self.buffer:
            return None
        events = list(self.buffer)
        data = ("\n".join(json.dumps(event, ensure_ascii=False, sort_keys=True, separators=(",", ":")) for event in events) + "\n").encode("utf-8")
        digest = hashlib.sha256(data).hexdigest()
        first_sequence, last_sequence = events[0]["sequence"], events[-1]["sequence"]
        chunk = {"path": f"events/{first_sequence:06d}-{last_sequence:06d}-{digest[:16]}.jsonl", "sha256": digest, "event_count": len(events), "first_sequence": first_sequence, "last_sequence": last_sequence, "created_at_utc": datetime.now(timezone.utc).isoformat()}
        self.pending = {"events": events, "data": data, "chunk": chunk}
        self._write_spool(self.pending)
        return self.pending
    def _manifest(self, chunks, final):
        sequences = [chunk["first_sequence"] for chunk in chunks] + [chunk["last_sequence"] for chunk in chunks]
        return {"schema_version": CONFIG["telemetry_schema_version"], "project": CONFIG["project"], "run_id": CONFIG["run_id"], "notebook_id": CONFIG["notebook_id"], "version": CONFIG["version"], "final": bool(final), "updated_at_utc": datetime.now(timezone.utc).isoformat(), "chunk_sha256_algorithm": "sha256", "chunk_count": len(chunks), "event_count": sum(chunk["event_count"] for chunk in chunks), "first_sequence": min(sequences) if sequences else None, "last_sequence": max(sequences) if sequences else None, "chunks": chunks}
    def _summary(self, chunks, terminal_status):
        manifest = self._manifest(chunks, True)
        return {"schema_version": CONFIG["telemetry_schema_version"], "project": CONFIG["project"], "run_id": CONFIG["run_id"], "notebook_id": CONFIG["notebook_id"], "version": CONFIG["version"], "terminal_status": terminal_status, "final": True, "finalized_at_utc": datetime.now(timezone.utc).isoformat(), "chunk_count": manifest["chunk_count"], "event_count": manifest["event_count"], "first_sequence": manifest["first_sequence"], "last_sequence": manifest["last_sequence"]}
    def _operation(self, relative_path, data):
        return self.commit_operation_add(path_in_repo=f"runs/{CONFIG['run_id']}/{relative_path}", path_or_fileobj=data)
    def _manifest_operation(self, chunks, final):
        data = json.dumps(self._manifest(chunks, final), ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
        return self._operation("manifest.json", data)
    def _summary_operation(self, chunks, terminal_status):
        data = json.dumps(self._summary(chunks, terminal_status), ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
        return self._operation("summary.json", data)
    def _commit(self, operations, commit_message, event_count):
        attempts = int(CONFIG["telemetry_retry_attempts"])
        for attempt in range(1, attempts + 1):
            try:
                self.api.create_commit(repo_id=self.repo_id, repo_type="dataset", token=self.token, commit_message=commit_message, operations=operations)
            except Exception as exc:
                self.failed_attempts += 1
                final_attempt = attempt == attempts
                print("COLAB_TELEMETRY " + json.dumps({"event": "upload_failed" if final_attempt else "upload_retry", "error_type": type(exc).__name__, "attempt": attempt, "attempts": attempts, "event_count": event_count, "events_retained": True}, sort_keys=True))
                if not final_attempt:
                    time.sleep(min(2 ** (attempt - 1), 4))
            else:
                return True
        return False
    def _commit_final_without_chunk(self, terminal_status):
        if self.finalized:
            return True
        operations = [self._manifest_operation(self.chunks, True), self._summary_operation(self.chunks, terminal_status)]
        if not self._commit(operations, f"telemetry {CONFIG['run_id']} final", 0):
            return False
        self.finalized = True
        return True
    def flush(self, final=False, terminal_status="unknown"):
        while True:
            pending = self._prepare_pending()
            if pending is None:
                return self._commit_final_without_chunk(terminal_status) if final else True
            events = pending["events"]
            if self.buffer[:len(events)] != events:
                raise RuntimeError("telemetry_buffer_integrity_error")
            is_final_chunk = final and len(self.buffer) == len(events)
            candidate_chunks = self.chunks + [pending["chunk"]]
            operations = [self._operation(pending["chunk"]["path"], pending["data"]), self._manifest_operation(candidate_chunks, is_final_chunk)]
            if is_final_chunk:
                operations.append(self._summary_operation(candidate_chunks, terminal_status))
            if not self._commit(operations, f"telemetry {CONFIG['run_id']} {pending['chunk']['first_sequence']:06d}-{pending['chunk']['last_sequence']:06d}", len(events)):
                return False
            self.chunks = candidate_chunks
            del self.buffer[:len(events)]
            self.pending = None
            self._cleanup_spool(pending)
            self.last_successful_flush_monotonic = time.monotonic()
            if is_final_chunk:
                self.finalized = True
                return True
            if not final:
                return True
    def close(self, terminal_status):
        return self.flush(final=True, terminal_status=terminal_status)

def maybe_mount_drive():
    trace("storage", "drive_not_mounted", reason="private_huggingface_repository_is_authoritative")
    return None

def run_synthetic_matrix_smoke(predictor):
    import torch
    torch.manual_seed(17)
    torch.backends.cuda.matmul.allow_tf32 = False
    device = torch.device("cuda:0")
    size = CONFIG["matrix_size"]
    left = torch.arange(size * size, device=device, dtype=torch.float32).reshape(size, size) / float(size * size)
    right = torch.flip(left, dims=(0, 1)) / float(size)
    result, elapsed_ms = left, []
    for iteration in range(1, CONFIG["matrix_iterations"] + 1):
        torch.cuda.synchronize(device)
        started = time.perf_counter()
        result = result @ right
        torch.cuda.synchronize(device)
        duration = round((time.perf_counter() - started) * 1000, 3)
        elapsed_ms.append(duration)
        trace("synthetic_smoke", "matrix_iteration", iteration=iteration, elapsed_ms=duration, checksum=round(float(result[:8, :8].sum().item()), 6), predictor=predictor.observe(iteration, duration))
    return {"status": "passed", "matrix_size": size, "iterations": CONFIG["matrix_iterations"], "elapsed_ms": elapsed_ms, "checksum": round(float(result[:8, :8].sum().item()), 6)}

def main():
    global TELEMETRY
    result, status, caught_error = None, "failed", None
    trace("notebook", "started", one_code_cell=True, publication_status=CONFIG["publication_status"], direct_colab_url=CONFIG["direct_colab_url"], recommended_gpu=CONFIG["recommended_gpu"], gpu_preference_order=CONFIG["gpu_preference_order"], ui_instruction="Runtime > Change runtime type > GPU")
    try:
        with traced("preflight"):
            verify_requested_runtime()
        with traced("secrets"):
            telemetry_token = load_secret(CONFIG["telemetry_secret_name"])
            trace("secrets", "loaded", secret_name=CONFIG["telemetry_secret_name"], value_logged=False)
        with traced("telemetry"):
            commit_operation_add, hf_api = ensure_huggingface_hub()
            TELEMETRY = HFTelemetry(CONFIG["telemetry_repository"], telemetry_token, commit_operation_add, hf_api)
            for record in TRACES:
                TELEMETRY.emit(record)
            trace("telemetry", "connected", transport="private_huggingface_dataset", repository_isolated=True, flush_interval_seconds=CONFIG["telemetry_flush_interval_seconds"])
        maybe_mount_drive()
        with traced("synthetic_smoke"):
            result = run_synthetic_matrix_smoke(MetricPredictor(mode="min"))
        status = "passed"
        trace("notebook", "completed", result=result, data_policy=CONFIG["data_policy"])
    except Exception as exc:
        caught_error = exc
        trace("notebook", "failed", "ERROR", error_type=type(exc).__name__)
        raise
    finally:
        telemetry_ok = True
        if TELEMETRY is not None:
            try:
                telemetry_ok = TELEMETRY.close(status)
            except Exception as exc:
                telemetry_ok = False
                print("COLAB_TELEMETRY " + json.dumps({"event": "close_failed", "error_type": type(exc).__name__}, sort_keys=True))
        if not telemetry_ok and caught_error is None:
            status = "telemetry_failed"
        print("COLAB_RESULT " + json.dumps({"status": status, "run_id": CONFIG["run_id"], "publication_status": CONFIG["publication_status"], "telemetry_configured": TELEMETRY is not None, "telemetry_complete": telemetry_ok, "result": result}, ensure_ascii=False, sort_keys=True))
        if not telemetry_ok and caught_error is None:
            raise RuntimeError("hf_telemetry_upload_failed")

main()
